In [0]:
# Bronze: raw Delta -> explode SDMX-JSON -> MERGE (incremental, idempotent)
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG      = "sample_project_playground_003"
RAW_TABLE    = f"{CATALOG}.raw.ecb_raw_json"
BRONZE_TABLE = f"{CATALOG}.bronze.ecb_fx_rates_daily"
CKPT         = f"/Volumes/{CATALOG}/raw/checkpoints/ecb_bronze_fx"

# --- DDL (idempotent) -------------------------------------------------------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
  frequency           STRING,
  currency            STRING,
  currency_denom      STRING,
  exchange_rate_type  STRING,
  series_variation    STRING,
  rate_date           DATE,
  exchange_rate       DOUBLE,
  obs_status          STRING,
  prepared_at         TIMESTAMP,
  source_dataset      STRING,
  source_file_path    STRING,
  bronze_ingested_at  TIMESTAMP
)
USING DELTA
CLUSTER BY (currency)
""")

COLS = ["frequency", "currency", "currency_denom", "exchange_rate_type",
        "series_variation", "rate_date", "exchange_rate", "obs_status",
        "prepared_at", "source_dataset", "source_file_path", "bronze_ingested_at"]

# --- SDMX-JSON document schema for from_json --------------------------------
# only what we need; unknown fields (attributes, links, names...) are ignored
PAYLOAD_SCHEMA = """
STRUCT<
  header: STRUCT<prepared: STRING>,
  dataSets: ARRAY<
    STRUCT<
      series: MAP<STRING, STRUCT<observations: MAP<STRING, ARRAY<DOUBLE>>>>
    >
  >,
  structure: STRUCT<
    dimensions: STRUCT<
      series: ARRAY<STRUCT<id: STRING, values: ARRAY<STRUCT<id: STRING>>>>,
      observation: ARRAY<STRUCT<values: ARRAY<STRUCT<id: STRING>>>>
    >,
    attributes: STRUCT<
      observation: ARRAY<STRUCT<id: STRING, values: ARRAY<STRUCT<id: STRING>>>>
    >
  >
>
"""

def build_bronze(df):
    d = (
        df.select(
            F.col("payload").cast("string").alias("json_str"),
            F.col("source_folder"),
            F.col("file_path").alias("source_file_path"),
        )
        .withColumn("doc", F.from_json(
            F.regexp_replace(F.col("json_str"), "^\ufeff", ""),  # defensive BOM strip
            PAYLOAD_SCHEMA))
        .select("source_folder", "source_file_path", "doc",
                F.col("doc.header.prepared").alias("prepared_str"),
                F.explode("doc.dataSets").alias("ds"))
        # series map: "0:0:0:0:0" -> struct(observations)
        .select("source_folder", "source_file_path", "doc", "prepared_str",
                F.explode_entries_fallback("ds.series") if False else
                F.explode(F.map_entries("ds.series")).alias("series_entry"))
        .select("source_folder", "source_file_path", "doc", "prepared_str",
                F.col("series_entry.key").alias("series_key"),
                F.explode(F.map_entries("series_entry.value.observations")).alias("obs_entry"))
        .select("source_folder", "source_file_path", "doc", "prepared_str",
                F.col("series_key"),
                F.col("obs_entry.key").alias("obs_idx"),
                F.col("obs_entry.value").alias("obs_arr"))
    )

    # --- observation date: obs index -> TIME_PERIOD value list --------------
    time_values = F.col("doc.structure.dimensions.observation")[0]["values"]
    rate_date = F.to_date(F.element_at(time_values, F.col("obs_idx").cast("int") + 1)["id"])

    # --- decode series key via structure.dimensions (generic) ---------------
    # key order == declared dimension order (SDMX spec)
    parts = F.split(F.col("series_key"), ":", -1)
    dim_arr = F.col("doc.structure.dimensions.series")

    def dim_val(i):
        seg = F.element_at(dim_arr, F.lit(i + 1))["values"]          # 1-based
        return F.element_at(seg, parts[i].cast("int") + 1)["id"]

    # --- observation status index -> id (e.g. 0 -> "A") ---------------------
    status_values = F.col("doc.structure.attributes.observation")[0]["values"]
    obs_status = F.element_at(status_values, F.col("obs_arr")[1].cast("int") + 1)["id"]

    return d.select(
        dim_val(0).alias("frequency"),
        dim_val(1).alias("currency"),
        dim_val(2).alias("currency_denom"),
        dim_val(3).alias("exchange_rate_type"),
        dim_val(4).alias("series_variation"),
        rate_date.alias("rate_date"),
        F.col("obs_arr")[0].alias("exchange_rate"),
        obs_status.alias("obs_status"),
        F.to_timestamp(F.col("prepared_str")).alias("prepared_at"),
        F.col("source_folder").alias("source_dataset"),
        F.col("source_file_path"),
        F.current_timestamp().alias("bronze_ingested_at"),
    ).where(rate_date.isNotNull() & F.col("obs_arr")[0].isNotNull())

def merge_bronze(batch_df, batch_id):
    n_raw = batch_df.count()
    staged = build_bronze(batch_df).cache()

    if n_raw > 0 and staged.isEmpty():
        staged.unpersist()
        raise RuntimeError(f"batch {batch_id}: {n_raw} raw rows but 0 bronze rows - parse failure")

    if staged.isEmpty():
        staged.unpersist()
        return

    sel = ", ".join(COLS)
    staged.createOrReplaceTempView("stg")
    spark.sql(f"""
      MERGE INTO {BRONZE_TABLE} t
      USING (
        SELECT {sel}
        FROM (
          SELECT *, ROW_NUMBER() OVER (
            PARTITION BY frequency, currency, currency_denom,
                         exchange_rate_type, series_variation, rate_date
            ORDER BY bronze_ingested_at DESC, source_file_path DESC
          ) AS rn
          FROM stg
        )
        WHERE rn = 1
      ) s
      ON  t.frequency          = s.frequency
      AND t.currency           = s.currency
      AND t.currency_denom     = s.currency_denom
      AND t.exchange_rate_type = s.exchange_rate_type
      AND t.series_variation   = s.series_variation
      AND t.rate_date          = s.rate_date
      WHEN MATCHED THEN UPDATE SET *
      WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"batch {batch_id}: raw={n_raw}, staged={staged.count()}")
    staged.unpersist()

# --- Run: incremental via checkpoint ----------------------------------------
query = (
    spark.readStream.format("delta")
    .table(RAW_TABLE)
    .writeStream
    .foreachBatch(merge_bronze)
    .option("checkpointLocation", CKPT)
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

# --- Bronze DQ ---------------------------------------------------------------
dups = spark.sql(f"""
  SELECT count(*) AS n FROM (
    SELECT 1 FROM {BRONZE_TABLE}
    GROUP BY frequency, currency, currency_denom,
             exchange_rate_type, series_variation, rate_date
    HAVING count(*) > 1)
""").collect()[0]["n"]
assert dups == 0, f"{dups} duplicate keys in bronze"

spark.sql(f"""
  SELECT count(*)                 AS rows_total,
         min(rate_date)           AS first_day,
         max(rate_date)           AS last_day,
         count(DISTINCT currency) AS currencies,
         min(exchange_rate)       AS min_rate,
         max(exchange_rate)       AS max_rate
  FROM {BRONZE_TABLE}
""").show()

spark.sql(f"""
  SELECT rate_date, exchange_rate, obs_status, source_dataset
  FROM {BRONZE_TABLE}
  ORDER BY rate_date DESC LIMIT 7
""").show()